# GenAI Text Detection — Fully From-Scratch, No scikit-learn

**Goal:** binary classification (human vs machine-generated text), evaluated by **macro F1**, with a target of **≥80% macro F1**, using only NumPy/Pandas/SciPy-sparse-as-storage — no `sklearn`, `xgboost`, or `lightgbm` anywhere. Every model, every metric, every train/val split below is hand-implemented.

## The journey

1. **Stylometric + word-frequency Bayesian hybrid** — hand-crafted features (word length, sentence structure, punctuation) plus a bag-of-words Naive Bayes, combined under one Bayesian posterior. Ceiling: **~0.64 macro F1**. Real signal, but too little information in a handful of aggregate features to compete with a rich text representation.
2. **Hand-rolled TF-IDF + hand-rolled logistic regression** — word 1-2grams + char 3-5grams TF-IDF (built by hand: term frequency, IDF, L2 normalization), trained with mini-batch gradient descent implemented from scratch. Result: **0.8236 macro F1** — matches a prior `sklearn` baseline (LinearSVC on the identical representation, 0.8229) almost exactly.
3. **Gradient-boosted decision trees from scratch** — the actual XGBoost algorithm (Newton leaf values, second-order split gain, vectorized split-finding) implemented from first principles, on a moderate dense feature set (stylometric + top-200 word counts). Result: **0.8320 ± 0.0052 (5-fold CV)**.

Both (2) and (3) clear the 80% target. This notebook reproduces all three, live, with real executed numbers — nothing here is copy-pasted from a prior run.

In [1]:
import time
import numpy as np
import pandas as pd

from ML_Project_NaiveBayes_Scratch import (
    SEED, DATA_DIR, OUTPUT_DIR, STOPWORDS,
    build_feature_matrix, FEATURE_NAMES,
    stratified_split, stratified_kfold, macro_f1,
    GaussianNaiveBayesScratch, MultinomialNaiveBayesScratch, HybridNaiveBayesScratch,
    select_decorrelated_features, build_vocabulary, build_count_matrix,
)

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
y_all = train_df["label"].to_numpy(dtype=int)
texts_all = train_df["text"].values
test_ids = test_df["id"].to_numpy()
test_texts = test_df["text"].values

print(f"train: {len(train_df)} rows | test: {len(test_df)} rows")
print(f"class balance: human={np.sum(y_all==0)} ({np.mean(y_all==0)*100:.1f}%) "
      f"machine={np.sum(y_all==1)} ({np.mean(y_all==1)*100:.1f}%)")

tr_idx, val_idx = stratified_split(y_all, val_frac=0.1, seed=SEED)
print(f"split: train={len(tr_idx)}  val={len(val_idx)}")

train: 20000 rows | test: 6999 rows
class balance: human=7496 (37.5%) machine=12504 (62.5%)
split: train=18000  val=2000


## Approach 1 — Bayesian Hybrid (stylometric + word-frequency)

Gaussian Naive Bayes over 9 decorrelated stylometric features (word length, sentence structure, punctuation density) combined via log-odds pooling with a Multinomial Naive Bayes over a small unigram+bigram vocabulary of content words. Best config found by 5-fold CV in an earlier iteration: `style_weight=0.5`, vocab variant `uni+bigram_content` at 1500 tokens, `alpha=1.0`, `var_smoothing=0.001`.

This is the **honest floor** of what a handful of interpretable features can do here — refit once below to confirm the number, not to re-tune it.

In [2]:
X_style_all = build_feature_matrix(texts_all)
decorr_idx = select_decorrelated_features(X_style_all[tr_idx], y_all[tr_idx], max_corr=0.6)
X_style_all_d = X_style_all[:, decorr_idx]

def content_filter(w):
    return w not in STOPWORDS

vocab_nb = build_vocabulary(texts_all[tr_idx], 1500, ngram_range=(1, 2), word_filter=content_filter)
C_tr = build_count_matrix(texts_all[tr_idx], vocab_nb, ngram_range=(1, 2), word_filter=content_filter)
C_val = build_count_matrix(texts_all[val_idx], vocab_nb, ngram_range=(1, 2), word_filter=content_filter)

hybrid = HybridNaiveBayesScratch(var_smoothing=0.001, alpha=1.0, style_weight=0.5)
hybrid.fit(X_style_all_d[tr_idx], C_tr, y_all[tr_idx])
p_val = hybrid.predict_proba(X_style_all_d[val_idx], C_val)[:, list(hybrid.classes_).index(1)]

best_t, best_f1 = 0.5, -1.0
for t in np.arange(0.25, 0.76, 0.01):
    f1 = macro_f1(y_all[val_idx], (p_val >= t).astype(int))
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print(f"Bayesian Hybrid: val macro F1 = {best_f1:.4f} (threshold={best_t:.2f})")
print("-> below the 80% target: aggregate stylometric + small-vocab features")
print("   carry real but limited signal. This is the expected floor, not a bug.")
bayes_f1 = best_f1

  extracted 14 features for 20000 docs in 3.5s


Bayesian Hybrid: val macro F1 = 0.6485 (threshold=0.31)
-> below the 80% target: aggregate stylometric + small-vocab features
   carry real but limited signal. This is the expected floor, not a bug.


## Approach 2 — Hand-rolled TF-IDF + Hand-rolled Logistic Regression

Word 1-2grams + char_wb 3-5grams, sublinear TF, smoothed IDF, L2-normalized rows, stored in `scipy.sparse` (a storage container, not a fitted algorithm). Logistic regression trained by explicit mini-batch gradient descent — sigmoid, log-loss gradient, L2 regularization, balanced class weighting, all written out by hand in `ML_Project_TFIDF_Scratch.py`.

This reproduces the representation that won the original (`sklearn`-based) baseline at 0.8229 — the entire point was to check whether the *representation*, not `sklearn`'s specific solver, was what mattered. It was.

In [3]:
from ML_Project_TFIDF_Scratch import (
    build_vocab_and_idf, vectorize, LinearClassifierScratch, best_threshold,
)

t0 = time.time()
vocab_tfidf, idf_tfidf, n_word = build_vocab_and_idf(texts_all[tr_idx], min_df=1)
X_tr_tfidf = vectorize(texts_all[tr_idx], vocab_tfidf, idf_tfidf)
X_val_tfidf = vectorize(texts_all[val_idx], vocab_tfidf, idf_tfidf)
print(f"vocab={len(vocab_tfidf)} (word={n_word}, char={len(vocab_tfidf)-n_word})  "
      f"X_tr={X_tr_tfidf.shape}  built in {time.time()-t0:.0f}s")

clf = LinearClassifierScratch(loss="logistic", reg=3e-5, lr=50.0, epochs=80).fit(
    X_tr_tfidf, y_all[tr_idx]
)
s_tr = clf.decision_function(X_tr_tfidf)
s_val = clf.decision_function(X_val_tfidf)
tr_f1 = macro_f1(y_all[tr_idx], (s_tr >= 0).astype(int))
thr, tfidf_f1 = best_threshold(s_val, y_all[val_idx])

print(f"\nTF-IDF + Logistic Regression: train F1={tr_f1:.4f}  val macro F1={tfidf_f1:.4f}  "
      f"(threshold={thr:+.3f}, gap={tr_f1-tfidf_f1:+.4f})")
print("-> clears the 80% target. Matches the sklearn baseline (0.8229) almost exactly,")
print("   confirming the representation (not sklearn's specific solver) is what mattered.")

vocab=640000 (word=120000, char=520000)  X_tr=(18000, 640000)  built in 23s



TF-IDF + Logistic Regression: train F1=0.9109  val macro F1=0.8236  (threshold=-0.080, gap=+0.0873)
-> clears the 80% target. Matches the sklearn baseline (0.8229) almost exactly,
   confirming the representation (not sklearn's specific solver) is what mattered.


## Approach 3 — Gradient-Boosted Decision Trees, from scratch

The real algorithm XGBoost/LightGBM run internally: a `DecisionTreeRegressor` that picks splits by maximizing the second-order (Newton) loss reduction, boosted by fitting each new tree to the logistic-loss gradient of the running prediction. Fed a **moderate, dense** feature set (14 stylometric + 4 extra + 200 word counts = 218 columns) rather than a huge sparse TF-IDF matrix — trees are known to underperform on very high-dimensional sparse input (confirmed earlier: `sklearn`'s RandomForest/XGBoost/LightGBM on the 5000-dim provided features scored only 0.70-0.75).

Best config from an earlier grid search (`learning_rate=0.3, max_depth=4, n_estimators=250, min_samples_leaf=20`). Re-validated here with fresh 5-fold CV — the honest headline, since a single holdout number is selection-biased after a hyperparameter search.

In [4]:
from ML_Project_GradientBoosting_Scratch import build_full_matrix, GradientBoostingClassifier

vocab_gb = build_vocabulary(texts_all[tr_idx], 200)
X_gb_all = build_full_matrix(texts_all, vocab_gb)
print(f"feature matrix: {X_gb_all.shape}")

fold_f1s = []
t0 = time.time()
for fi, (k_tr, k_val) in enumerate(stratified_kfold(y_all, k=5, seed=SEED)):
    gb = GradientBoostingClassifier(
        n_estimators=250, learning_rate=0.3, max_depth=4,
        min_samples_leaf=20, reg_lambda=1.0, colsample=0.6, seed=SEED,
    ).fit(X_gb_all[k_tr], y_all[k_tr])
    f1 = macro_f1(y_all[k_val], gb.predict(X_gb_all[k_val]))
    fold_f1s.append(f1)
    print(f"  fold {fi+1}: val macro F1 = {f1:.4f}")

gb_cv_mean, gb_cv_std = float(np.mean(fold_f1s)), float(np.std(fold_f1s))
print(f"\nGradient Boosting: 5-fold CV macro F1 = {gb_cv_mean:.4f} +/- {gb_cv_std:.4f}  "
      f"({time.time()-t0:.0f}s)")
print("-> clears the 80% target, and is CV-averaged (more robust than a single holdout).")

# Single-holdout number too, for an apples-to-apples row in the summary table below
# (same split as Approaches 1 and 2).
gb_holdout = GradientBoostingClassifier(
    n_estimators=250, learning_rate=0.3, max_depth=4,
    min_samples_leaf=20, reg_lambda=1.0, colsample=0.6, seed=SEED,
).fit(X_gb_all[tr_idx], y_all[tr_idx])
gb_tr_f1 = macro_f1(y_all[tr_idx], gb_holdout.predict(X_gb_all[tr_idx]))
gb_val_f1 = macro_f1(y_all[val_idx], gb_holdout.predict(X_gb_all[val_idx]))
print(f"Same split as Approaches 1-2: train F1={gb_tr_f1:.4f}  val F1={gb_val_f1:.4f}  "
      f"gap={gb_tr_f1-gb_val_f1:+.4f}")

  extracted 14 features for 20000 docs in 3.5s


feature matrix: (20000, 218)


  fold 1: val macro F1 = 0.8363


  fold 2: val macro F1 = 0.8268


  fold 3: val macro F1 = 0.8260


  fold 4: val macro F1 = 0.8317


  fold 5: val macro F1 = 0.8392

Gradient Boosting: 5-fold CV macro F1 = 0.8320 +/- 0.0052  (185s)
-> clears the 80% target, and is CV-averaged (more robust than a single holdout).


Same split as Approaches 1-2: train F1=0.9517  val F1=0.8352  gap=+0.1165


## Final comparison and honest verdict

In [5]:
summary = pd.DataFrame([
    {"approach": "1. Bayesian Hybrid (stylometric + word-freq)",
     "val_macro_f1": bayes_f1, "protocol": "single 90/10 split", "hits_80pct": bayes_f1 >= 0.80},
    {"approach": "2. Hand-rolled TF-IDF + Logistic Regression",
     "val_macro_f1": tfidf_f1, "protocol": "single 90/10 split", "hits_80pct": tfidf_f1 >= 0.80},
    {"approach": "3. Gradient Boosted Trees (same split)",
     "val_macro_f1": gb_val_f1, "protocol": "single 90/10 split", "hits_80pct": gb_val_f1 >= 0.80},
    {"approach": "3. Gradient Boosted Trees (5-fold CV)",
     "val_macro_f1": gb_cv_mean, "protocol": "5-fold CV mean", "hits_80pct": gb_cv_mean >= 0.80},
])
print(summary.to_string(index=False))

print("\n" + "="*72)
print("VERDICT")
print("="*72)
print(f"Target: >= 0.80 macro F1, fully from scratch (no sklearn/xgboost/lightgbm).")
print(f"Result: TWO independent from-scratch approaches clear it:")
print(f"  - TF-IDF + Logistic Regression: {tfidf_f1:.4f}")
print(f"  - Gradient Boosted Trees:       {gb_cv_mean:.4f} (CV) / {gb_val_f1:.4f} (this split)")
print(f"The Bayesian hybrid ({bayes_f1:.4f}) does not, and isn't expected to — it")
print(f"deliberately uses far less information (a handful of aggregate features).")
print()
print("Caveat worth flagging honestly: the TRAIN/VAL gap is larger for gradient")
print(f"boosting (+{gb_tr_f1-gb_val_f1:.3f}) than for the linear TF-IDF model (+{tr_f1-tfidf_f1:.3f}).")
print("A prior sklearn model on this same TF-IDF representation dropped from 0.82")
print("validation to 0.73 on the actual Kaggle leaderboard (train/test distribution")
print("shift) -- the model with the bigger train/val gap is the one more likely to")
print("drop further on unseen data. Recommend TF-IDF+LogReg as the primary submission")
print("(same representation as the proven 0.7299 leaderboard result) and treat the")
print("gradient boosting model as a strong second opinion / ensemble candidate,")
print("not a blind replacement.")

                                    approach  val_macro_f1           protocol  hits_80pct
1. Bayesian Hybrid (stylometric + word-freq)      0.648461 single 90/10 split       False
 2. Hand-rolled TF-IDF + Logistic Regression      0.823610 single 90/10 split        True
      3. Gradient Boosted Trees (same split)      0.835161 single 90/10 split        True
       3. Gradient Boosted Trees (5-fold CV)      0.832018     5-fold CV mean        True

VERDICT
Target: >= 0.80 macro F1, fully from scratch (no sklearn/xgboost/lightgbm).
Result: TWO independent from-scratch approaches clear it:
  - TF-IDF + Logistic Regression: 0.8236
  - Gradient Boosted Trees:       0.8320 (CV) / 0.8352 (this split)
The Bayesian hybrid (0.6485) does not, and isn't expected to — it
deliberately uses far less information (a handful of aggregate features).

Caveat worth flagging honestly: the TRAIN/VAL gap is larger for gradient
boosting (+0.117) than for the linear TF-IDF model (+0.087).
A prior sklearn model o

## Refit winners on full data and save test predictions

In [6]:
# TF-IDF + Logistic Regression -- refit on all 20K rows (vocab/IDF refit too, no leakage).
t0 = time.time()
vocab_full, idf_full, _ = build_vocab_and_idf(texts_all, min_df=1)
X_full_tfidf = vectorize(texts_all, vocab_full, idf_full)
X_test_tfidf = vectorize(test_texts, vocab_full, idf_full)
clf_final = LinearClassifierScratch(loss="logistic", reg=3e-5, lr=50.0, epochs=80).fit(
    X_full_tfidf, y_all
)
y_test_tfidf = (clf_final.decision_function(X_test_tfidf) >= thr).astype(int)
pd.DataFrame({"id": test_ids, "label": y_test_tfidf}).to_csv(
    OUTPUT_DIR / "TFIDF_Scratch_Prediction.csv", index=False
)
print(f"TF-IDF+LogReg refit + predicted in {time.time()-t0:.0f}s  "
      f"(machine={y_test_tfidf.sum()}, human={(y_test_tfidf==0).sum()})")

# Gradient Boosting -- refit on all 20K rows.
t0 = time.time()
vocab_gb_full = build_vocabulary(texts_all, 200)
X_gb_full = build_full_matrix(texts_all, vocab_gb_full)
X_gb_test = build_full_matrix(test_texts, vocab_gb_full)
gb_final = GradientBoostingClassifier(
    n_estimators=250, learning_rate=0.3, max_depth=4,
    min_samples_leaf=20, reg_lambda=1.0, colsample=0.6, seed=SEED,
).fit(X_gb_full, y_all)
y_test_gb = gb_final.predict(X_gb_test)
pd.DataFrame({"id": test_ids, "label": y_test_gb}).to_csv(
    OUTPUT_DIR / "GradientBoosting_Scratch_Prediction.csv", index=False
)
print(f"Gradient Boosting refit + predicted in {time.time()-t0:.0f}s  "
      f"(machine={y_test_gb.sum()}, human={(y_test_gb==0).sum()})")

print("\nBoth prediction files saved to predictions/. Recommended primary submission:")
print("predictions/TFIDF_Scratch_Prediction.csv")

TF-IDF+LogReg refit + predicted in 40s  (machine=5382, human=1617)


  extracted 14 features for 20000 docs in 3.6s


  extracted 14 features for 6999 docs in 1.5s


Gradient Boosting refit + predicted in 56s  (machine=4890, human=2109)

Both prediction files saved to predictions/. Recommended primary submission:
predictions/TFIDF_Scratch_Prediction.csv
